# Acquire Valued Shoppers Challenge


<img src='https://incontextsolutions.com/wp-content/uploads/2023/12/value-shopper-scaled.jpg'>


Bu çalışmada `Acquire Valued Shoppers Challenge` yarışması için teklif geçmişi ve alışveriş kayıtları kullanılarak müşterinin tekrar alışveriş yapma olasılığını tahmin eden bir sınıflandırma modeli oluşturulmuştur.


## Akış

1. Kütüphaneleri yükleme
2. Drive bağlama ve zip içeriğini tanımlama
3. Ana dosyaları yükleme
4. Transaction verisini parça parça hazırlama
5. Özellik çıkarımı
6. Model kurma
7. ROC AUC değerlendirmesi
8. Test tahmini ve submission
9. Sonuç


## 1. Kütüphaneleri Yükleme


In [1]:
# Bu bölümde veri hazırlama, parça parça dosya okuma ve sınıflandırma modeli için gerekli kütüphaneleri içe aktarıyoruz.


In [2]:
from google.colab import drive
import os
import zipfile
import gzip

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


## 2. Drive Bağlama ve Zip İçeriğini Tanımlama


In [3]:
# Bu bölümde yarışma zip dosyasını Drive üzerinden bağlıyor ve gerekli dosyaları zip içinden buluyoruz.


In [4]:
drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/Colab Data Dosyaları/acquire-valued-shoppers-challenge.zip'
print('Zip dosyası:', zip_path)
print('Zip mevcut mu?:', os.path.exists(zip_path))

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_members = zip_ref.namelist()

print('Zip içindeki ilk dosyalar:', zip_members[:20])

def find_zip_member(members, filename):
    for member in members:
        if member.endswith(filename):
            return member
    raise FileNotFoundError(f'{filename} zip içinde bulunamadi.')

train_history_member = find_zip_member(zip_members, 'trainHistory.csv.gz')
test_history_member = find_zip_member(zip_members, 'testHistory.csv.gz')
offers_member = find_zip_member(zip_members, 'offers.csv.gz')
transactions_member = find_zip_member(zip_members, 'transactions.csv.gz')
sample_submission_member = find_zip_member(zip_members, 'sampleSubmission.csv.gz')

print('Train history member:', train_history_member)
print('Test history member:', test_history_member)
print('Offers member:', offers_member)
print('Transactions member:', transactions_member)
print('Sample submission member:', sample_submission_member)


Mounted at /content/drive
Zip dosyası: /content/drive/MyDrive/Colab Data Dosyaları/acquire-valued-shoppers-challenge.zip
Zip mevcut mu?: True
Zip içindeki ilk dosyalar: ['offers.csv.gz', 'sampleSubmission.csv.gz', 'testHistory.csv.gz', 'trainHistory.csv.gz', 'transactions.csv.gz']
Train history member: trainHistory.csv.gz
Test history member: testHistory.csv.gz
Offers member: offers.csv.gz
Transactions member: transactions.csv.gz
Sample submission member: sampleSubmission.csv.gz


## 3. Ana Dosyaları Yükleme


In [5]:
# Bu bölümde küçük ve orta boy dosyaları doğrudan zip içinden açıyoruz.


In [6]:
def read_gzip_csv_from_zip(zip_path, member_name, **kwargs):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        with zip_ref.open(member_name) as raw_file:
            with gzip.open(raw_file, 'rt') as gz_file:
                return pd.read_csv(gz_file, **kwargs)

train_history = read_gzip_csv_from_zip(zip_path, train_history_member)
test_history = read_gzip_csv_from_zip(zip_path, test_history_member)
offers = read_gzip_csv_from_zip(zip_path, offers_member)
sample_submission = read_gzip_csv_from_zip(zip_path, sample_submission_member)

print('Train history shape:', train_history.shape)
print('Test history shape:', test_history.shape)
print('Offers shape:', offers.shape)
print('Sample submission shape:', sample_submission.shape)
train_history.head()


Train history shape: (160057, 7)
Test history shape: (151484, 5)
Offers shape: (37, 6)
Sample submission shape: (151484, 2)


,id,chain,offer,market,repeattrips,repeater,offerdate
0,86246,205,1208251,34,5,t,2013-04-24
1,86252,205,1197502,34,16,t,2013-03-27
2,12682470,18,1197502,11,0,f,2013-03-28
3,12996040,15,1197502,9,0,f,2013-03-25
4,13089312,15,1204821,9,0,f,2013-04-01


In [7]:
# Bu yarışmada asıl büyük dosya transactions.csv.gz olduğu için onu ayrı ve kontrollü bir şekilde işleyeceğiz.


## 4. Transaction Verisini Parça Parça Hazırlama


In [8]:
# Bu bölümde çok büyük transaction dosyasını tek seferde değil, parçalar halinde okuyup yalnızca tekliflerle ilgili kayıtları tutuyoruz.


In [9]:
offer_categories = set(offers['category'].astype(str))
offer_companies = set(offers['company'].astype(str))
offer_brands = set(offers['brand'].astype(str))

relevant_customer_ids = set(train_history['id']).union(set(test_history['id']))

transaction_usecols = [
    'id', 'chain', 'dept', 'category', 'company', 'brand',
    'date', 'purchasequantity', 'purchaseamount'
]

chunksize = 500000
max_chunks = 8
filtered_chunks = []
processed_chunks = 0

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    with zip_ref.open(transactions_member) as raw_file:
        with gzip.open(raw_file, 'rt') as gz_file:
            chunk_iter = pd.read_csv(gz_file, usecols=transaction_usecols, chunksize=chunksize)
            for chunk in chunk_iter:
                processed_chunks += 1

                chunk['category'] = chunk['category'].astype(str)
                chunk['company'] = chunk['company'].astype(str)
                chunk['brand'] = chunk['brand'].astype(str)

                mask_customer = chunk['id'].isin(relevant_customer_ids)
                mask_offer_match = (
                    chunk['category'].isin(offer_categories) |
                    chunk['company'].isin(offer_companies) |
                    chunk['brand'].isin(offer_brands)
                )
                chunk = chunk.loc[mask_customer & mask_offer_match].copy()

                if not chunk.empty:
                    filtered_chunks.append(chunk)

                if processed_chunks >= max_chunks:
                    break

transactions = pd.concat(filtered_chunks, ignore_index=True) if filtered_chunks else pd.DataFrame(columns=transaction_usecols)
transactions['date'] = pd.to_datetime(transactions['date'], errors='coerce')

print('İşlenen chunk sayısı:', processed_chunks)
print('Filtered transactions shape:', transactions.shape)
transactions.head()


İşlenen chunk sayısı: 8
Filtered transactions shape: (374431, 9)


,id,chain,dept,category,company,brand,date,purchasequantity,purchaseamount
0,86246,205,99,9909,104538848,15343,2012-03-02,1,2.49
1,86246,205,21,2106,105100050,27873,2012-03-02,1,3.29
2,86246,205,9,907,101410010,13791,2012-03-02,1,3.99
3,86246,205,26,2630,103700030,14647,2012-03-02,1,1.00
4,86246,205,58,5824,108674585,55172,2012-03-02,1,3.29


In [10]:
# Burada tüm transaction verisini zorlamıyoruz. Yarışmanın mantığını kurmak için ilgili müşteri ve teklif kayıtlarının anlamlı bir kısmını kullanıyoruz.


## 5. Özellik Çıkarımı


In [11]:
# Bu bölümde transaction verisinden müşteri bazlı özetler çıkarıp teklif ve geçmiş tabloları ile birleştiriyoruz.


In [12]:
customer_features = transactions.groupby('id').agg(
    transaction_count=('purchaseamount', 'count'),
    total_purchase_amount=('purchaseamount', 'sum'),
    average_purchase_amount=('purchaseamount', 'mean'),
    max_purchase_amount=('purchaseamount', 'max'),
    total_purchase_quantity=('purchasequantity', 'sum'),
    unique_categories=('category', 'nunique'),
    unique_companies=('company', 'nunique'),
    unique_brands=('brand', 'nunique'),
    last_purchase_date=('date', 'max')
).reset_index()

reference_date = transactions['date'].max()
customer_features['days_since_last_purchase'] = (reference_date - customer_features['last_purchase_date']).dt.days
customer_features = customer_features.drop(columns=['last_purchase_date'])

train_df = train_history.merge(offers, on='offer', how='left').merge(customer_features, on='id', how='left')
test_df = test_history.merge(offers, on='offer', how='left').merge(customer_features, on='id', how='left')

train_df['target'] = (train_df['repeater'] == 't').astype(int)

print('Train merged shape:', train_df.shape)
print('Test merged shape:', test_df.shape)
train_df.head()


Train merged shape: (160057, 22)
Test merged shape: (151484, 19)


,id,chain,offer,market,repeattrips,repeater,offerdate,category,quantity,company,offervalue,brand,transaction_count,total_purchase_amount,average_purchase_amount,max_purchase_amount,total_purchase_quantity,unique_categories,unique_companies,unique_brands,days_since_last_purchase,target
0,86246,205,1208251,34,5,t,2013-04-24,2202,1,104460040,2.00,3718,1006.0,4385.61,4.359453,27.96,1334.0,104.0,79.0,143.0,91.0,1
1,86252,205,1197502,34,16,t,2013-03-27,3203,1,106414464,0.75,13474,957.0,4419.08,4.617638,79.92,1396.0,91.0,72.0,131.0,119.0,1
2,12682470,18,1197502,11,0,f,2013-03-28,3203,1,106414464,0.75,13474,70.0,343.66,4.909429,22.98,79.0,23.0,20.0,30.0,132.0,0
3,12996040,15,1197502,9,0,f,2013-03-25,3203,1,106414464,0.75,13474,29.0,128.64,4.435862,17.99,31.0,9.0,11.0,13.0,125.0,0
4,13089312,15,1204821,9,0,f,2013-04-01,5619,1,107717272,1.50,102504,79.0,354.16,4.483038,30.00,93.0,19.0,21.0,27.0,119.0,0


## 6. Model Kurma


In [13]:
# Bu bölümde sayısal ve kategorik alanları hazırlayıp müşterinin tekrar alışveriş yapma olasılığını tahmin eden başlangıç modelini kuruyoruz.


In [14]:
feature_columns = [
    'chain', 'market', 'offerdate', 'category', 'quantity', 'company', 'offervalue', 'brand',
    'transaction_count', 'total_purchase_amount', 'average_purchase_amount', 'max_purchase_amount',
    'total_purchase_quantity', 'unique_categories', 'unique_companies', 'unique_brands',
    'days_since_last_purchase'
]

x = train_df[feature_columns].copy()
y = train_df['target'].copy()

x['offerdate'] = pd.to_datetime(x['offerdate'], errors='coerce')
x['offer_year'] = x['offerdate'].dt.year
x['offer_month'] = x['offerdate'].dt.month
x['offer_day'] = x['offerdate'].dt.day
x = x.drop(columns=['offerdate'])

x_train, x_valid, y_train, y_valid = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

categorical_features = ['chain', 'market', 'category', 'company', 'brand']
numerical_features = [col for col in x.columns if col not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median'))
        ]), numerical_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_features)
    ]
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_split=10,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced_subsample'
    ))
])

model.fit(x_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['quantity', 'offervalue',
                                                   'transaction_count',
                                                   'total_purchase_amount',
                                                   'average_purchase_amount',
                                                   'max_purchase_amount',
                                                   'total_purchase_quantity',
                                                   'unique_categories',
                                                   'unique_companies',
                                                   'unique_brands',
                                                   'days_since_last_p...
                                                   'offer_year', 'offer_month',
                                                   'offer_day']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['chain', 'market',
                                                   'category', 'company',
                                                   'brand'])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced_subsample',
                                        max_depth=12, min_samples_split=10,
                                        n_estimators=200, n_jobs=-1,
                                        random_state=42))])

## 7. ROC AUC Değerlendirmesi


In [15]:
# Bu bölümde modelin ayrıştırma gücünü ROC AUC metriği ile ölçüyoruz.


In [16]:
valid_probs = model.predict_proba(x_valid)[:, 1]
roc_auc = roc_auc_score(y_valid, valid_probs)
print('Validation ROC AUC:', round(roc_auc, 5))


Validation ROC AUC: 0.69181


## 8. Test Tahmini ve Submission


In [17]:
# Bu bölümde test seti için tekrar alışveriş yapma olasılığını hesaplayıp submission dosyasını oluşturuyoruz.


In [18]:
test_x = test_df[feature_columns].copy()
test_x['offerdate'] = pd.to_datetime(test_x['offerdate'], errors='coerce')
test_x['offer_year'] = test_x['offerdate'].dt.year
test_x['offer_month'] = test_x['offerdate'].dt.month
test_x['offer_day'] = test_x['offerdate'].dt.day
test_x = test_x.drop(columns=['offerdate'])

test_probs = model.predict_proba(test_x)[:, 1]

submission = sample_submission.copy()
submission['repeatProbability'] = test_probs

print('Submission shape:', submission.shape)
submission.head()


Submission shape: (151484, 2)


,id,repeatProbability
0,12262064,0.499125
1,12277270,0.516104
2,12332190,0.463031
3,12524696,0.464838
4,13074629,0.481450


In [19]:
output_path = '/content/acquire_submission.csv'
submission.to_csv(output_path, index=False)
print('Kaydedilen dosya:', output_path)


Kaydedilen dosya: /content/acquire_submission.csv


## 9. Sonuç


Bu çalışmada Acquire Valued Shoppers Challenge yarışması için teklif bilgileri ve alışveriş geçmişi birleştirilerek müşterinin tekrar alışveriş yapma olasılığını tahmin eden bir başlangıç modeli oluşturuldu. Elde edilen sonuçlara göre model validation verisi üzerinde 0.69181 ROC AUC değeri elde etti ve test müşterileri için tekrar alışveriş olasılıkları üretildi.
